#  Segmentation de Clientèle par Machine Learning (K-Means)

---

## Introduction au Projet

Ce notebook présente une **analyse complète de segmentation de clientèle** utilisant l'algorithme de **clustering K-Means**. La segmentation de clientèle est une technique marketing essentielle qui permet de :

-  **Identifier des groupes homogènes** de clients partageant des caractéristiques similaires
-  **Personnaliser les stratégies marketing** pour chaque segment
-  **Optimiser l'allocation des ressources** marketing
-  **Améliorer la satisfaction client** grâce à des offres ciblées

### Objectifs de l'Analyse

1. Explorer et comprendre les données clients
2. Appliquer l'algorithme K-Means pour segmenter la clientèle
3. Créer des profils détaillés pour chaque segment
4. Formuler des recommandations marketing actionnables
5. Développer un modèle prédictif pour classifier de nouveaux clients

### Méthodologie

Nous suivrons une approche structurée en **7 étapes** :

```
Chargement → Exploration → Prétraitement → Clustering → Analyse → Prédiction → Rapport
```

---

##  1. Import des Bibliothèques

Nous commençons par importer toutes les bibliothèques nécessaires pour notre analyse :

- **pandas** et **numpy** : manipulation et calcul sur les données
- **matplotlib** et **seaborn** : visualisation des données
- **scikit-learn** : algorithmes de machine learning et prétraitement
- **warnings** : gestion des avertissements pour un affichage propre

In [1]:
# Importation des bibliothèques essentielles
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Configuration esthétique des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Bibliothèques importées avec succès !")

---

##  2. Configuration de l'Environnement

Création d'un dossier dédié pour stocker tous les résultats de l'analyse (graphiques, rapports, fichiers CSV).

In [ ]:
# Création du dossier de résultats
from pathlib import Path
# Racine du projet : fonctionne depuis notebooks/ ou depuis la racine du dépôt
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PATH = ROOT / "data" / "customer_segmentation_mixed.csv"
RESULTATS_DIR = str(ROOT / "resultats")
os.makedirs(RESULTATS_DIR, exist_ok=True)

print(f" Dossier de résultats créé : {RESULTATS_DIR}/")

---

##  3. Définition des Fonctions Utilitaires

### 3.1 Fonction de Chargement des Données

Cette fonction tente de charger le fichier CSV de données clients. Si le fichier n'existe pas, elle génère automatiquement des données synthétiques pour l'analyse.

**Variables du dataset** :
- `Age` : Âge du client (18-70 ans)
- `Annual_Income` : Revenu annuel en dollars
- `Spending_Score` : Score de dépense (1-100)
- `Monthly_Visits` : Nombre de visites mensuelles
- `Gender` : Genre (Male/Female)
- `Preferred_Product` : Catégorie de produit préférée

In [ ]:
def charger_donnees():
    """Charger les données clients depuis un fichier CSV ou générer des données synthétiques"""
    try:
        data = pd.read_csv(DATA_PATH)
        print(f" Données chargées depuis '{DATA_PATH.name}'")
        return data
    except FileNotFoundError:
        print(" Fichier non trouvé. Génération de données synthétiques...")
        return generer_donnees_echantillon()

def generer_donnees_echantillon():
    """Générer des données synthétiques réalistes pour l'analyse"""
    np.random.seed(42)  # Pour la reproductibilité
    n_samples = 500
    
    # Variables numériques
    age = np.random.randint(18, 70, n_samples)
    annual_income = np.random.normal(60000, 20000, n_samples).clip(20000, 150000)
    spending_score = np.random.randint(1, 100, n_samples)
    monthly_visits = np.random.poisson(4, n_samples).clip(1, 20)
    
    # Variables catégoriques
    gender = np.random.choice(['Male', 'Female'], n_samples, p=[0.48, 0.52])
    preferred_product = np.random.choice(
        ['Electronics', 'Clothing', 'Home', 'Beauty', 'Sports'], 
        n_samples, 
        p=[0.3, 0.25, 0.2, 0.15, 0.1]
    )
    
    data = pd.DataFrame({
        'Age': age,
        'Annual_Income': annual_income,
        'Spending_Score': spending_score,
        'Monthly_Visits': monthly_visits,
        'Gender': gender,
        'Preferred_Product': preferred_product
    })
    
    # Sauvegarde
    chemin_fichier = os.path.join(RESULTATS_DIR, "donnees_clients_generes.csv")
    data.to_csv(chemin_fichier, index=False)
    print(f" Données synthétiques générées et sauvegardées : {chemin_fichier}")
    
    return data

### 3.2 Fonction de Prétraitement

Le prétraitement est une étape cruciale qui prépare les données pour l'algorithme de clustering :

1. **Nettoyage** : suppression des valeurs manquantes
2. **Encodage** : transformation des variables catégoriques en valeurs numériques
   - `Gender` → 0 ou 1
   - `Preferred_Product` → 0, 1, 2, 3, ou 4

In [ ]:
def preprocess_data(data):
    """Prétraitement des données : nettoyage et encodage"""
    print("\n" + "="*60)
    print("🔧 PRÉTRAITEMENT DES DONNÉES")
    print("="*60)
    
    data_clean = data.copy()
    
    # Informations initiales
    print(f"Nombre initial de clients : {len(data_clean)}")
    
    # Gestion des valeurs manquantes
    valeurs_manquantes = data_clean.isnull().sum().sum()
    if valeurs_manquantes > 0:
        print(f" Suppression de {valeurs_manquantes} valeurs manquantes")
        data_clean = data_clean.dropna()
        print(f"Nombre de clients après nettoyage : {len(data_clean)}")
    else:
        print(" Aucune valeur manquante détectée")
    
    # Encodage des variables catégoriques
    le_gender = LabelEncoder()
    le_product = LabelEncoder()
    
    data_clean['Gender_encoded'] = le_gender.fit_transform(data_clean['Gender'])
    data_clean['Preferred_Product_encoded'] = le_product.fit_transform(data_clean['Preferred_Product'])
    
    print(" Prétraitement terminé")
    
    return data_clean, le_gender, le_product

### 3.3 Fonction de Clustering K-Means

#### Qu'est-ce que K-Means ?

K-Means est un algorithme de **clustering non supervisé** qui :
1. Divise les données en K groupes (clusters)
2. Minimise la variance intra-cluster
3. Maximise la variance inter-cluster

#### Étapes de l'algorithme :
1. Initialisation de K centroïdes aléatoires
2. Attribution de chaque point au centroïde le plus proche
3. Recalcul des centroïdes comme moyenne des points assignés
4. Répétition jusqu'à convergence

#### Métriques d'évaluation :
- **Score Silhouette** (0 à 1) : mesure la cohésion et la séparation des clusters
  - > 0.5 : bonne séparation
  - 0.2-0.5 : séparation acceptable
  - < 0.2 : mauvaise séparation
  
- **Davies-Bouldin Index** : mesure la similarité entre clusters
  - Plus proche de 0 = meilleur

In [ ]:
def perform_clustering(data, n_clusters=4):
    """Exécuter l'algorithme K-Means et évaluer les résultats"""
    print(f"\n Exécution du clustering K-Means avec {n_clusters} clusters...")
    
    # Sélection des features pour le clustering
    features = ["Age", "Annual_Income", "Spending_Score", "Monthly_Visits"]
    
    # Normalisation (StandardScaler)
    # Pourquoi normaliser ? Pour que toutes les variables aient le même poids
    # (éviter que Annual_Income domine à cause de sa grande échelle)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data[features])
    
    # Application de K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    
    # Calcul des scores d'évaluation
    silhouette = silhouette_score(X_scaled, clusters)
    db_index = davies_bouldin_score(X_scaled, clusters)
    
    # Ajout des clusters aux données
    data_with_clusters = data.copy()
    data_with_clusters['Cluster'] = clusters
    
    return {
        'scaler': scaler,
        'kmeans': kmeans,
        'clusters': clusters,
        'features_scaled': X_scaled,
        'silhouette_score': silhouette,
        'davies_bouldin_score': db_index,
        'features': features,
        'data_with_clusters': data_with_clusters
    }

### 3.4 Fonction de Création des Profils de Clusters

Cette fonction analyse chaque cluster pour créer un **profil détaillé** comprenant :
- Statistiques démographiques (âge, genre)
- Comportement d'achat (revenu, dépenses, fréquence)
- Préférences produits
- Description qualitative du segment

In [ ]:
def create_cluster_profiles(data, clusters):
    """Créer des profils détaillés pour chaque cluster"""
    data_with_clusters = data.copy()
    data_with_clusters['Cluster'] = clusters
    
    profiles = {}
    
    for cluster in sorted(data_with_clusters['Cluster'].unique()):
        cluster_data = data_with_clusters[data_with_clusters['Cluster'] == cluster]
        
        profile = {
            'Taille': len(cluster_data),
            'Pourcentage': len(cluster_data) / len(data_with_clusters) * 100,
            'Age_moyen': cluster_data['Age'].mean(),
            'Revenu_moyen': cluster_data['Annual_Income'].mean(),
            'Score_depense_moyen': cluster_data['Spending_Score'].mean(),
            'Visites_moyennes': cluster_data['Monthly_Visits'].mean(),
            'Genre_majoritaire': cluster_data['Gender'].mode()[0] if not cluster_data['Gender'].mode().empty else 'N/A',
            'Genre_Female_pourcentage': (cluster_data['Gender'] == 'Female').mean() * 100,
            'Genre_Male_pourcentage': (cluster_data['Gender'] == 'Male').mean() * 100,
            'Produit_prefere': cluster_data['Preferred_Product'].mode()[0] if not cluster_data['Preferred_Product'].mode().empty else 'N/A',
            'Description': ''
        }
        
        # Détermination automatique de la description du cluster
        income = profile['Revenu_moyen']
        spending = profile['Score_depense_moyen']
        visits = profile['Visites_moyennes']
        
        if income > data['Annual_Income'].quantile(0.75) and spending > data['Spending_Score'].quantile(0.75):
            profile['Description'] = 'Clients Premium - Haute valeur'
        elif visits > data['Monthly_Visits'].quantile(0.75):
            profile['Description'] = 'Clients Fidèles - Visites fréquentes'
        elif spending < data['Spending_Score'].quantile(0.25):
            profile['Description'] = 'Clients Économes - Dépenses faibles'
        else:
            profile['Description'] = 'Clients Standards - Comportement moyen'
        
        profiles[cluster] = profile
    
    return profiles

### 3.5 Fonction d'Analyse Exploratoire

L'**analyse exploratoire des données (EDA)** est essentielle avant tout modèle de machine learning. Elle permet de :
- Comprendre la structure des données
- Détecter les anomalies et valeurs aberrantes
- Identifier les relations entre variables
- Informer les décisions de prétraitement

In [ ]:
def analyser_donnees(data):
    """Analyse exploratoire des données"""
    print("\n" + "="*60)
    print(" ANALYSE EXPLORATOIRE DES DONNÉES")
    print("="*60)
    
    # Informations de base
    print(f"\n INFORMATIONS GÉNÉRALES")
    print(f"Nombre de clients : {len(data)}")
    print(f"Nombre de variables : {len(data.columns)}")
    
    print(f"\n APERÇU DES DONNÉES")
    print(data.head())
    
    print(f"\n STATISTIQUES DESCRIPTIVES")
    print(data.describe())
    
    print(f"\n TYPES DE DONNÉES")
    print(data.dtypes)
    
    print(f"\n VALEURS MANQUANTES")
    missing = data.isnull().sum()
    for col, count in missing.items():
        if count > 0:
            print(f"  {col}: {count} valeurs ({count/len(data)*100:.1f}%)")
        else:
            print(f"  {col}: Aucune valeur manquante")
    
    return data

### 3.6 Fonction de Visualisation des Données

Les visualisations permettent de :
1. **Distributions** : comprendre la répartition des variables
2. **Corrélations** : identifier les relations entre variables
3. **Scatter plots** : visualiser les relations multivariées
4. **Analyses catégorielles** : comprendre la composition démographique

In [ ]:
def visualiser_donnees(data):
    """Générer les visualisations des données"""
    print("\n GÉNÉRATION DES VISUALISATIONS...")
    
    # 1. Distribution des variables numériques
    num_cols = ["Age", "Annual_Income", "Spending_Score", "Monthly_Visits"]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()
    
    for idx, col in enumerate(num_cols):
        axes[idx].hist(data[col], bins=30, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution de {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Fréquence')
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    chemin_image = os.path.join(RESULTATS_DIR, 'distribution_variables.png')
    plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
    print(f"   {chemin_image}")
    plt.close()
    
    # 2. Analyse des variables catégoriques
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    gender_counts = data['Gender'].value_counts()
    axes[0].bar(gender_counts.index, gender_counts.values, color=['lightblue', 'lightpink'])
    axes[0].set_title('Répartition par genre')
    axes[0].set_ylabel('Nombre')
    for i, v in enumerate(gender_counts.values):
        axes[0].text(i, v + 5, str(v), ha='center')
    
    product_counts = data['Preferred_Product'].value_counts()
    axes[1].barh(range(len(product_counts)), product_counts.values)
    axes[1].set_yticks(range(len(product_counts)))
    axes[1].set_yticklabels(product_counts.index)
    axes[1].set_title('Produits préférés')
    axes[1].set_xlabel('Nombre')
    
    plt.tight_layout()
    chemin_image = os.path.join(RESULTATS_DIR, 'analyse_categorielle.png')
    plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
    print(f"   {chemin_image}")
    plt.close()
    
    # 3. Scatter plot revenu vs dépenses
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(data['Annual_Income'], data['Spending_Score'], 
                         c=data['Age'], cmap='viridis', alpha=0.6)
    plt.colorbar(scatter, label='Age')
    plt.xlabel('Revenu Annuel ($)')
    plt.ylabel('Score de Dépense')
    plt.title('Revenu vs Score de Dépense (couleur = Age)')
    plt.grid(True, alpha=0.3)
    chemin_image = os.path.join(RESULTATS_DIR, 'revenu_vs_depenses.png')
    plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
    print(f"   {chemin_image}")
    plt.close()
    
    # 4. Matrice de corrélation
    numeric_data = data.select_dtypes(include=[np.number])
    if len(numeric_data.columns) > 1:
        corr_matrix = numeric_data.corr()
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                   square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title('Matrice de corrélation')
        plt.tight_layout()
        chemin_image = os.path.join(RESULTATS_DIR, 'matrice_correlation.png')
        plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
        print(f"   {chemin_image}")
        plt.close()

### 3.7 Fonction d'Exécution du Clustering avec Visualisations

Cette fonction combine le clustering avec plusieurs visualisations clés :

1. **Visualisation PCA 2D** : réduction dimensionnelle pour voir les clusters en 2D
2. **Revenu vs Dépenses** : visualisation business-oriented des segments
3. **Méthode du coude** : détermination du nombre optimal de clusters
4. **Score Silhouette** : validation de la qualité du clustering

In [ ]:
def executer_clustering(data_clean, n_clusters=4):
    """Exécuter le clustering et générer les visualisations"""
    print("\n" + "="*60)
    print(" CLUSTERING K-MEANS")
    print("="*60)
    
    # Exécution du clustering
    results = perform_clustering(data_clean, n_clusters)
    data_with_clusters = results['data_with_clusters']
    
    # Affichage des métriques
    print(f"\n MÉTRIQUES D'ÉVALUATION")
    print(f"  Score Silhouette : {results['silhouette_score']:.3f} (idéal > 0.5)")
    print(f"  Score Davies-Bouldin : {results['davies_bouldin_score']:.3f} (idéal proche de 0)")
    print(f"  Inertie : {results['kmeans'].inertia_:.0f}")
    
    print("\n GÉNÉRATION DES VISUALISATIONS DES CLUSTERS...")
    
    # 1. Visualisation 2D avec PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(results['features_scaled'])
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                         c=data_with_clusters['Cluster'], cmap='tab10', s=50, alpha=0.6)
    plt.xlabel('Composante Principale 1')
    plt.ylabel('Composante Principale 2')
    plt.title('Visualisation 2D des clusters (PCA)')
    plt.colorbar(scatter, label='Cluster')
    plt.grid(True, alpha=0.3)
    chemin_image = os.path.join(RESULTATS_DIR, 'clusters_pca.png')
    plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
    print(f"   {chemin_image}")
    plt.close()
    
    # 2. Revenu vs Score de dépense par cluster
    plt.figure(figsize=(10, 8))
    
    clusters_unique = sorted(data_with_clusters['Cluster'].unique())
    colors = plt.cm.tab10(np.linspace(0, 1, len(clusters_unique)))
    
    for i, cluster in enumerate(clusters_unique):
        cluster_data = data_with_clusters[data_with_clusters['Cluster'] == cluster]
        plt.scatter(cluster_data['Annual_Income'], cluster_data['Spending_Score'],
                   label=f'Cluster {cluster}', color=colors[i], s=50, alpha=0.6)
    
    # Centroïdes
    centroids = results['scaler'].inverse_transform(results['kmeans'].cluster_centers_)
    plt.scatter(centroids[:, 1], centroids[:, 2],
               marker='X', s=200, c='black', label='Centroïdes', edgecolors='red', linewidths=2)
    
    plt.xlabel('Revenu Annuel ($)')
    plt.ylabel('Score de Dépense')
    plt.title('Segmentation Clientèle : Revenu vs Score de Dépense')
    plt.legend()
    plt.grid(True, alpha=0.3)
    chemin_image = os.path.join(RESULTATS_DIR, 'clusters_revenu_depenses.png')
    plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
    print(f"   {chemin_image}")
    plt.close()
    
    # 3. Méthode du coude
    print("\n DÉTERMINATION DU NOMBRE OPTIMAL DE CLUSTERS...")
    
    wcss = []
    silhouette_scores = []
    k_range = range(2, 11)
    
    for k in k_range:
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        clusters_temp = kmeans_temp.fit_predict(results['features_scaled'])
        wcss.append(kmeans_temp.inertia_)
        silhouette_scores.append(silhouette_score(results['features_scaled'], clusters_temp))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(k_range, wcss, 'bo-')
    ax1.set_xlabel('Nombre de clusters')
    ax1.set_ylabel('WCSS (Inertie)')
    ax1.set_title('Méthode du coude')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(k_range, silhouette_scores, 'ro-')
    ax2.set_xlabel('Nombre de clusters')
    ax2.set_ylabel('Score Silhouette')
    ax2.set_title('Score Silhouette')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    chemin_image = os.path.join(RESULTATS_DIR, 'methode_coude.png')
    plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
    print(f"   {chemin_image}")
    plt.close()
    
    optimal_k = k_range[np.argmax(silhouette_scores)]
    print(f"\n Nombre optimal de clusters recommandé : {optimal_k}")
    
    return data_with_clusters, results

### 3.8 Fonction d'Analyse des Profils

Cette fonction génère :
1. **Profils détaillés** de chaque segment
2. **Recommandations marketing** spécifiques à chaque cluster
3. **Visualisations comparatives** entre clusters
4. **Insights actionnables** pour les équipes business

In [ ]:
def analyser_profils(data_with_clusters, results):
    """Analyser les profils des clusters"""
    print("\n" + "="*60)
    print(" ANALYSE DES PROFILS DES CLUSTERS")
    print("="*60)
    
    # Création des profils
    profiles = create_cluster_profiles(data_with_clusters, results['clusters'])
    
    # Affichage des profils détaillés
    print("\n PROFILS DÉTAILLÉS DES CLUSTERS")
    print("="*60)
    
    for cluster, profile in profiles.items():
        print(f"\n{'='*50}")
        print(f" CLUSTER {cluster} - {profile['Description']}")
        print(f"{'='*50}")
        print(f"\n Statistiques :")
        print(f"  • Taille : {profile['Taille']} clients ({profile['Pourcentage']:.1f}%)")
        print(f"  • Âge moyen : {profile['Age_moyen']:.1f} ans")
        print(f"  • Revenu moyen : ${profile['Revenu_moyen']:,.0f}")
        print(f"  • Score dépense moyen : {profile['Score_depense_moyen']:.1f}/100")
        print(f"  • Visites/mois : {profile['Visites_moyennes']:.1f}")
        
        print(f"\n Démographie :")
        print(f"  • Genre majoritaire : {profile['Genre_majoritaire']}")
        print(f"  • Femmes : {profile['Genre_Female_pourcentage']:.1f}%")
        print(f"  • Hommes : {profile['Genre_Male_pourcentage']:.1f}%")
        print(f"  • Produit préféré : {profile['Produit_prefere']}")
        
        print(f"\n Recommandations marketing :")
        if "Premium" in profile['Description']:
            print("  1. Offres VIP et produits exclusifs")
            print("  2. Service client personnalisé")
            print("  3. Invitations à des événements spéciaux")
            print("  4. Programmes de fidélité premium")
        elif "Fidèles" in profile['Description']:
            print("  1. Programmes de fidélité avec récompenses")
            print("  2. Offres de cross-selling")
            print("  3. Communication régulière (newsletters)")
            print("  4. Programmes de parrainage")
        elif "Économes" in profile['Description']:
            print("  1. Promotions ciblées")
            print("  2. Offres de première commande")
            print("  3. Produits en soldes")
            print("  4. Échantillons gratuits")
        else:
            print("  1. Campagnes marketing générales")
            print("  2. Offres groupées")
            print("  3. Communication modérée")
            print("  4. Enquêtes de satisfaction")
    
    # Tableau comparatif
    print(f"\n{'='*60}")
    print(" TABLEAU COMPARATIF DES CLUSTERS")
    print(f"{'='*60}")
    
    comparative_df = pd.DataFrame(profiles).T
    display_df = comparative_df[['Taille', 'Pourcentage', 'Age_moyen', 'Revenu_moyen', 
                               'Score_depense_moyen', 'Visites_moyennes', 'Description']].copy()
    display_df.columns = ['Taille', '%', 'Âge moy', 'Revenu moy ($)', 'Score dép', 'Visites/mois', 'Description']
    
    pd.set_option('display.float_format', lambda x: f'{x:.1f}')
    print(display_df.to_string())
    
    # Sauvegarde des résultats
    chemin_fichier = os.path.join(RESULTATS_DIR, 'resultats_clustering.csv')
    data_with_clusters.to_csv(chemin_fichier, index=False)
    print(f"\n Résultats sauvegardés : {chemin_fichier}")
    
    # Visualisations comparatives
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    sizes = [profile['Taille'] for profile in profiles.values()]
    labels = [f'Cluster {i}' for i in profiles.keys()]
    axes[0, 0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    axes[0, 0].set_title('Répartition des clusters')
    
    incomes = [profile['Revenu_moyen'] for profile in profiles.values()]
    axes[0, 1].bar(range(len(incomes)), incomes)
    axes[0, 1].set_xticks(range(len(incomes)))
    axes[0, 1].set_xticklabels(labels)
    axes[0, 1].set_title('Revenu moyen par cluster')
    axes[0, 1].set_ylabel('Revenu ($)')
    
    scores = [profile['Score_depense_moyen'] for profile in profiles.values()]
    axes[1, 0].bar(range(len(scores)), scores)
    axes[1, 0].set_xticks(range(len(scores)))
    axes[1, 0].set_xticklabels(labels)
    axes[1, 0].set_title('Score de dépense moyen par cluster')
    axes[1, 0].set_ylabel('Score')
    
    visits = [profile['Visites_moyennes'] for profile in profiles.values()]
    axes[1, 1].bar(range(len(visits)), visits)
    axes[1, 1].set_xticks(range(len(visits)))
    axes[1, 1].set_xticklabels(labels)
    axes[1, 1].set_title('Visites moyennes par cluster')
    axes[1, 1].set_ylabel('Visites/mois')
    
    plt.tight_layout()
    chemin_image = os.path.join(RESULTATS_DIR, 'comparaison_clusters.png')
    plt.savefig(chemin_image, dpi=300, bbox_inches='tight')
    print(f"   {chemin_image}")
    plt.close()
    
    return profiles

### 3.9 Fonction de Prédiction pour Nouveaux Clients

Une fois le modèle entraîné, nous pouvons **classifier de nouveaux clients** dans les segments existants.

#### Processus de prédiction :
1. Normalisation des features du nouveau client (avec le même scaler)
2. Prédiction du cluster avec le modèle K-Means
3. Calcul de la confiance basée sur la distance au centroïde
4. Recommandations basées sur le profil du cluster

In [ ]:
def predire_nouveau_client(data_clean, results, profiles):
    """Prédire le cluster pour un nouveau client"""
    print("\n" + "="*60)
    print(" PRÉDICTION POUR UN NOUVEAU CLIENT")
    print("="*60)
    
    # Exemple de nouveau client
    print("\n EXEMPLE DE NOUVEAU CLIENT :")
    
    customer_features = {
        "Age": 35,
        "Annual_Income": 75000,
        "Spending_Score": 75,
        "Monthly_Visits": 8,
        "Gender": "Male",
        "Preferred_Product": "Electronics"
    }
    
    print(f"  • Âge : {customer_features['Age']}")
    print(f"  • Revenu annuel : ${customer_features['Annual_Income']:,.0f}")
    print(f"  • Score de dépense : {customer_features['Spending_Score']}/100")
    print(f"  • Visites mensuelles : {customer_features['Monthly_Visits']}")
    print(f"  • Genre : {customer_features['Gender']}")
    print(f"  • Produit préféré : {customer_features['Preferred_Product']}")
    
    # Préparation des données
    customer_df = pd.DataFrame([customer_features])
    features = ["Age", "Annual_Income", "Spending_Score", "Monthly_Visits"]
    X_customer = customer_df[features]
    
    # Normalisation et prédiction
    X_scaled = results['scaler'].transform(X_customer)
    cluster = results['kmeans'].predict(X_scaled)[0]
    
    # Calcul de la confiance
    distances = results['kmeans'].transform(X_scaled)[0]
    confidence = 1 - (distances[cluster] / distances.max())
    
    print(f"\n RÉSULTAT DE LA PRÉDICTION :")
    print(f"  → Ce client appartient au Cluster {cluster}")
    print(f"  → Confiance de prédiction : {confidence*100:.1f}%")
    
    if cluster in profiles:
        profile = profiles[cluster]
        print(f"\n CARACTÉRISTIQUES DU CLUSTER {cluster} :")
        print(f"  → Description : {profile['Description']}")
        print(f"  → Taille : {profile['Taille']} clients ({profile['Pourcentage']:.1f}%)")
    
    return customer_features, cluster, confidence

### 3.10 Fonction de Génération du Rapport

Génération d'un **rapport complet au format texte** contenant :
- Résumé exécutif
- Métriques de performance
- Profils détaillés des segments
- Recommandations marketing actionnables

In [ ]:
def sauvegarder_rapport(data_clean, results, profiles, n_clusters):
    """Sauvegarder un rapport complet des résultats"""
    print("\n" + "="*60)
    print(" GÉNÉRATION DU RAPPORT COMPLET")
    print("="*60)
    
    chemin_rapport = os.path.join(RESULTATS_DIR, "rapport_clustering.txt")
    
    with open(chemin_rapport, 'w', encoding='utf-8') as f:
        f.write("="*70 + "\n")
        f.write("RAPPORT DE SEGMENTATION DE CLIENTÈLE\n")
        f.write("="*70 + "\n\n")
        
        f.write("1. RÉSUMÉ EXÉCUTIF\n")
        f.write("-"*40 + "\n")
        f.write(f"Date d'analyse : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Nombre total de clients : {len(data_clean)}\n")
        f.write(f"Nombre de clusters : {n_clusters}\n")
        f.write(f"Score Silhouette : {results['silhouette_score']:.3f}\n")
        f.write(f"Score Davies-Bouldin : {results['davies_bouldin_score']:.3f}\n\n")
        
        f.write("2. RÉPARTITION DES CLUSTERS\n")
        f.write("-"*40 + "\n")
        cluster_counts = results['data_with_clusters']['Cluster'].value_counts().sort_index()
        for cluster, count in cluster_counts.items():
            percentage = count / len(data_clean) * 100
            desc = profiles[cluster]['Description'] if cluster in profiles else "N/A"
            f.write(f"Cluster {cluster}: {count} clients ({percentage:.1f}%) - {desc}\n")
        f.write("\n")
        
        f.write("3. PROFILS DÉTAILLÉS DES CLUSTERS\n")
        f.write("-"*40 + "\n")
        for cluster, profile in profiles.items():
            f.write(f"\n[CLUSTER {cluster}] - {profile['Description']}\n")
            f.write(f"  Taille : {profile['Taille']} clients ({profile['Pourcentage']:.1f}%)\n")
            f.write(f"  Âge moyen : {profile['Age_moyen']:.1f} ans\n")
            f.write(f"  Revenu moyen : ${profile['Revenu_moyen']:,.0f}\n")
            f.write(f"  Score dépense : {profile['Score_depense_moyen']:.1f}/100\n")
            f.write(f"  Visites/mois : {profile['Visites_moyennes']:.1f}\n")
            f.write(f"  Genre majoritaire : {profile['Genre_majoritaire']}\n")
            f.write(f"  Produit préféré : {profile['Produit_prefere']}\n")
        f.write("\n")
        
        f.write("4. RECOMMANDATIONS MARKETING\n")
        f.write("-"*40 + "\n")
        for cluster, profile in profiles.items():
            f.write(f"\nCluster {cluster} ({profile['Description']}):\n")
            if "Premium" in profile['Description']:
                f.write("  - Offres VIP et produits exclusifs\n")
                f.write("  - Service client personnalisé\n")
                f.write("  - Invitations événements spéciaux\n")
            elif "Fidèles" in profile['Description']:
                f.write("  - Programmes de fidélité\n")
                f.write("  - Offres de cross-selling\n")
                f.write("  - Communication régulière\n")
            elif "Économes" in profile['Description']:
                f.write("  - Promotions ciblées\n")
                f.write("  - Offres de première commande\n")
                f.write("  - Produits en soldes\n")
            else:
                f.write("  - Campagnes générales\n")
                f.write("  - Offres groupées\n")
                f.write("  - Communication modérée\n")
    
    print(f" Rapport complet sauvegardé : {chemin_rapport}")

---

##  4. Exécution du Pipeline Complet

Maintenant que toutes les fonctions sont définies, nous pouvons exécuter le pipeline complet d'analyse en **7 étapes séquentielles**.

###  ÉTAPE 1 : Chargement des Données

Chargement du fichier CSV ou génération de données synthétiques si nécessaire.

In [ ]:
# Configuration
n_clusters = 4  # Nombre de segments à créer

print("\n" + "="*70)
print(" SEGMENTATION DE CLIENTÈLE PAR MACHINE LEARNING (K-MEANS)")
print("="*70)
print(f"\nTous les résultats seront sauvegardés dans : {RESULTATS_DIR}/")
print(f"Nombre de clusters : {n_clusters}")

# Chargement des données
print(f"\n{'='*50}")
print(" ÉTAPE 1 : CHARGEMENT DES DONNÉES")
print(f"{'='*50}")
data = charger_donnees()

###  ÉTAPE 2 : Analyse Exploratoire

Exploration des données pour comprendre leur structure et identifier les patterns initiaux.

In [ ]:
print(f"\n{'='*50}")
print(" ÉTAPE 2 : ANALYSE EXPLORATOIRE")
print(f"{'='*50}")
analyser_donnees(data)
visualiser_donnees(data)

###  ÉTAPE 3 : Prétraitement

Nettoyage et transformation des données pour le machine learning.

In [ ]:
print(f"\n{'='*50}")
print(" ÉTAPE 3 : PRÉTRAITEMENT")
print(f"{'='*50}")
data_clean, le_gender, le_product = preprocess_data(data)

###  ÉTAPE 4 : Clustering K-Means

Application de l'algorithme K-Means pour segmenter la clientèle.

In [ ]:
print(f"\n{'='*50}")
print(" ÉTAPE 4 : CLUSTERING K-MEANS")
print(f"{'='*50}")
data_with_clusters, results = executer_clustering(data_clean, n_clusters)

###  ÉTAPE 5 : Analyse des Profils

Création de profils détaillés pour chaque segment avec recommandations marketing.

In [ ]:
print(f"\n{'='*50}")
print(" ÉTAPE 5 : ANALYSE DES PROFILS")
print(f"{'='*50}")
profiles = analyser_profils(data_with_clusters, results)

###  ÉTAPE 6 : Prédiction (Exemple)

Démonstration de la classification d'un nouveau client.

In [ ]:
print(f"\n{'='*50}")
print(" ÉTAPE 6 : PRÉDICTION (EXEMPLE)")
print(f"{'='*50}")
customer, cluster_pred, confidence = predire_nouveau_client(data_clean, results, profiles)

###  ÉTAPE 7 : Génération du Rapport

Sauvegarde d'un rapport complet avec tous les résultats.

In [ ]:
sauvegarder_rapport(data_clean, results, profiles, n_clusters)

---

##  5. Résumé Final

Affichage du résumé des résultats et des fichiers générés.

In [ ]:
# Résumé final
print(f"\n{'='*70}")
print(" ANALYSE TERMINÉE AVEC SUCCÈS !")
print(f"{'='*70}")

# Lister les fichiers générés
fichiers_generes = os.listdir(RESULTATS_DIR)
fichiers_images = [f for f in fichiers_generes if f.endswith('.png')]
fichiers_csv = [f for f in fichiers_generes if f.endswith('.csv')]
fichiers_txt = [f for f in fichiers_generes if f.endswith('.txt')]

print(f"\n FICHIERS GÉNÉRÉS DANS LE DOSSIER '{RESULTATS_DIR}/' :")

if fichiers_images:
    print(f"\n VISUALISATIONS ({len(fichiers_images)} fichiers):")
    for img in sorted(fichiers_images):
        print(f"  • {img}")

if fichiers_csv:
    print(f"\n📈 DONNÉES ({len(fichiers_csv)} fichiers):")
    for csv in sorted(fichiers_csv):
        print(f"  • {csv}")

if fichiers_txt:
    print(f"\n📄 RAPPORTS ({len(fichiers_txt)} fichiers):")
    for txt in sorted(fichiers_txt):
        print(f"  • {txt}")

print(f"\n📊 RÉSUMÉ DES RÉSULTATS :")
print(f"  • Nombre de clients analysés : {len(data_clean)}")
print(f"  • Nombre de clusters créés : {n_clusters}")
print(f"  • Score Silhouette : {results['silhouette_score']:.3f}")

# Afficher les clusters
cluster_counts = data_with_clusters['Cluster'].value_counts().sort_index()
print(f"\n🎯 RÉPARTITION DES CLUSTERS :")
for cluster, count in cluster_counts.items():
    percentage = count / len(data_with_clusters) * 100
    if cluster in profiles:
        description = profiles[cluster]['Description']
        print(f"  Cluster {cluster}: {count} clients ({percentage:.1f}%) - {description}")
    else:
        print(f"  Cluster {cluster}: {count} clients ({percentage:.1f}%)")

print(f"\n{'='*70}")
print("✨ Analyse terminée avec succès !")
print(f"Tous les résultats sont disponibles dans le dossier '{RESULTATS_DIR}/'")
print(f"{'='*70}")

---

##  6. Conclusion et Perspectives

###  Ce que nous avons accompli :

1.  **Segmentation réussie** de la clientèle en groupes homogènes
2.  **Création de profils détaillés** pour chaque segment
3.  **Formulation de recommandations marketing** actionnables
4.  **Développement d'un modèle prédictif** pour nouveaux clients

###  Applications business :

- **Marketing personnalisé** : campagnes ciblées par segment
- **Optimisation des ressources** : allocation budget selon la valeur client
- **Amélioration de la rétention** : programmes de fidélité adaptés
- **Développement produit** : offres alignées avec les préférences

###  Pistes d'amélioration :

1. **Ajouter plus de variables** : historique d'achats, satisfaction, channel préféré
2. **Tester d'autres algorithmes** : DBSCAN, Hierarchical Clustering, Gaussian Mixture
3. **Analyse temporelle** : évolution des segments dans le temps
4. **Scoring RFM** : Récence, Fréquence, Montant
5. **Intégration CRM** : automatisation de la classification en temps réel



---

**Projet réalisé par Hiba Gourri dans le cadre d'un projet  universitaire**  
**Algorithme** : K-Means Clustering  
**Technologies** : Python, Scikit-learn, Pandas, Matplotlib, Seaborn